## Exploración de los documentos reales del Ayuntamiento

Objetivo de este notebook: entender la forma real de los datos ANTES de
diseñar nada. No se define ningún tipo, no se transforma nada — solo
observación y conclusiones, igual que en el EDA de HateSpeech-Detector.

Fuentes:
- financiero_2025_indicadores_control_financiero.xlsx (7 hojas)
- agencia_innovacion_y_empleo_principales_avances.xlsx
- financiero_informe_1q.docx

In [ ]:
from pathlib import Path
import openpyxl

DATA_DIR = Path.cwd().parent / "data" / "raw"

excel_path = DATA_DIR / "financiero_2025_indicadores_control_financiero.xlsx"
assert excel_path.exists(), f"No encuentro el archivo en {excel_path}"

wb = openpyxl.load_workbook(excel_path, data_only=True)
print("Hojas encontradas:", wb.sheetnames)

In [ ]:
ws = wb["Empleo"]
for row in ws.iter_rows(min_row=1, max_row=4, values_only=True):
    print(row)

### Hallazgo 1: cabecera a dos niveles

Fila 2 = cabecera de GRUPO fusionada ("SERVICIO DE ORIENTACIÓN...").
Fila 3 = cabecera real de columna ("TOTAL CONTRATOS", etc.).
Un extractor que tome "la primera fila con contenido" como cabecera
asigna mal todo lo que viene después.

In [ ]:
for sheet in wb.sheetnames:
    ws = wb[sheet]
    print(f"{sheet:30s} | dims: {ws.dimensions} | celdas fusionadas: {len(ws.merged_cells.ranges)}")

### Hallazgo 2: no es solo una hoja

7 hojas en el mismo archivo, cada una con su propia estructura de
cabecera y su propio número de celdas fusionadas. El extractor no
puede asumir una estructura fija por archivo, ni siquiera por hoja.

In [ ]:
import docx

docx_path = DATA_DIR / "financiero_informe_1q.docx"
assert docx_path.exists(), f"No encuentro el archivo en {docx_path}"

d = docx.Document(docx_path)
print("Párrafos con texto:", sum(1 for p in d.paragraphs if p.text.strip()))
print("Tablas:", len(d.tables))
print()
print("Ejemplo de párrafo narrativo con cifra incrustada:")
print(repr(d.paragraphs[6].text))

In [ ]:
print("Ejemplo de tabla suelta (primeras 3 filas):")
for row in d.tables[0].rows[:3]:
    print([c.text for c in row.cells])

### Hallazgo 3: texto narrativo con cifras incrustadas

El .docx no es solo tablas — hay cifras dentro de frases normales
("1.396 personas"). El Agente Analista tiene que saber extraer números
de texto corrido, no solo de celdas de tabla.

## Conclusiones del notebook 00

1. Cabeceras a uno o dos niveles, según hoja — no se puede asumir fija.
2. Múltiples hojas por archivo, cada una con su propia estructura.
3. Cifras tanto en tablas como incrustadas en texto narrativo.
4. El diseño BloqueContenido (texto/tabla genérico, sin categorías
   fijas) que ya teníais en extractor_generico.py sigue siendo correcto
   — estos hallazgos lo confirman, no lo cambian.

Con esto ya podemos pasar a 01_shared_state.ipynb.

## Conclusiones del notebook 00

1. **Cabecera a uno o dos niveles, y varía por hoja, no por archivo.**
   Empleo, Cesión espacios, Emprendimiento y Jornadas y Programas tienen
   cabeceras fusionadas (grupo + columna). Centro Formación e Incidencias
   informáticas NO — cabecera simple de una fila. El extractor no puede
   asumir un único patrón para todo el libro; tiene que evaluar cada hoja
   por separado.

2. **7 hojas por archivo, cada una con estructura propia** (dimensiones,
   nº de celdas fusionadas y temática distintas). Confirma que la ingesta
   debe operar hoja a hoja, no libro a libro.

3. **El .docx tiene dos formas de dato distintas:**
   - Texto narrativo con cifras incrustadas en la frase
     (ej. "1.396 personas"), no extraíble como tabla.
   - Tablas sueltas con celdas de texto vacías que representan
     continuidad de valor (fila 2 con MES='' significa "sigue en
     Septiembre"), NO fusión real de Excel — no se resuelve con
     openpyxl, hay que tratarlo como una regla de "rellenar hacia
     abajo" sobre texto vacío.

4. **Confirma (no cambia) el diseño BloqueContenido** ya construido en
   extractor_generico.py: unidad genérica de texto o tabla, sin
   categorías fijas, procesada hoja a hoja / bloque a bloque.

5. **Nuevo, no estaba anotado antes:** hacen falta DOS estrategias de
   "relleno" distintas — propagación de celdas fusionadas (Excel,
   ya resuelta con openpyxl) y relleno de celdas de texto vacío en
   tablas de Word (pendiente, no la teníais cubierta).